Архитектура:
- **Retrieval**: гибридный поиск BM25 + TF-IDF (через word bigrams) + TF-IDF (char 3-5 grams) с лемматизацией (pymorphy3)
- **Grounding**: фильтрация чанков по порогу релевантности, сборка контекста
- **Generation**: google/gemma-4-31b через локальный vllm сервинг
- **Метрика**: BERT-Recall-L (оптимизируем длину ответа)

Я пользовался кластером HSE CHARIZMA, использовав 1 A100 80Gb VRAM

In [1]:
import csv
import re
import statistics
import time
from pathlib import Path
from typing import List, Dict, Tuple

import faiss
import numpy as np
import openai
import pymorphy3
import torch
from jinja2 import Template
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

csv.field_size_limit(10 ** 7)


In [2]:
def load_csv(path: Path) -> List[Dict]:
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return list(reader)
questions = load_csv(Path("questions.csv"))
websites  = load_csv(Path("websites.csv"))


In [3]:
morph = pymorphy3.MorphAnalyzer()
_lemma_cache: Dict[str, str] = {}

def lemmatize(text: str) -> str:
    text = re.sub(r"[^а-яёa-z0-9\s]", " ", text.lower())
    words = text.split()
    result = []
    for w in words:
        if len(w) < 2:
            continue
        if w in _lemma_cache:
            result.append(_lemma_cache[w])
        else:
            parsed = morph.parse(w)
            lemma = parsed[0].normal_form if parsed else w
            _lemma_cache[w] = lemma
            result.append(lemma)
    return " ".join(result)


In [4]:
def clean_text(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def make_chunks(
    doc_id: str,
    url: str,
    title: str,
    text: str,
    chunk_size: int = 150,
    overlap: int = 30,
) -> List[Dict]:
    text = clean_text(text)
    words = text.split()
    if not words:
        return []

    title_prefix = clean_text(title) + " " if title else ""

    chunks = []
    i = 0
    chunk_idx = 0
    while i < len(words):
        raw_text = " ".join(words[i : i + chunk_size])
        text_for_index = (title_prefix + raw_text) if chunk_idx == 0 else raw_text
        chunks.append(
            {
                "chunk_id":   f"{doc_id}__{chunk_idx}",
                "doc_id":     doc_id,
                "url":        url,
                "title":      title,
                "text":       raw_text,
                "index_text": text_for_index,
                "lemma_text": lemmatize(text_for_index),
            }
        )
        chunk_idx += 1
        i += chunk_size - overlap
    return chunks


all_chunks: List[Dict] = []
for doc in tqdm(websites):
    text = doc.get("text", "")
    if len(text) < 20:
        continue
    all_chunks.extend(make_chunks(
        doc_id=doc["web_id"],
        url=doc.get("url", ""),
        title=doc.get("title", ""),
        text=text,
    ))


100%|██████████| 1937/1937 [00:09<00:00, 212.94it/s]


In [5]:
lemma_texts = [c["lemma_text"] for c in all_chunks]

tokenized_corpus = [text.split() for text in lemma_texts]
bm25_index = BM25Okapi(tokenized_corpus)

tfidf_word = TfidfVectorizer(min_df=1, max_df=0.95, ngram_range=(1, 2), sublinear_tf=True)
M_word = tfidf_word.fit_transform(lemma_texts)

tfidf_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_df=0.95, sublinear_tf=True)
M_char = tfidf_char.fit_transform(lemma_texts)


In [6]:
embed_device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model  = SentenceTransformer("ai-forever/FRIDA", device=embed_device)
embed_dim    = embed_model.get_sentence_embedding_dimension()

faiss_index_path = Path("faiss_frida.index")
chunks_path      = Path("chunks_meta.npy")

if faiss_index_path.exists() and chunks_path.exists():
    faiss_index  = faiss.read_index(str(faiss_index_path))
    chunks_order = np.load(str(chunks_path), allow_pickle=True).tolist()
else:
    embeddings = embed_model.encode(
        [c["text"] for c in all_chunks],
        prompt_name="search_document",
        batch_size=128,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    faiss_index = faiss.IndexFlatIP(embed_dim)
    faiss_index.add(embeddings.astype(np.float32))
    faiss.write_index(faiss_index, str(faiss_index_path))
    chunks_order = [c["chunk_id"] for c in all_chunks]
    np.save(str(chunks_path), np.array(chunks_order, dtype=object))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/509 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/54.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.68M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/5.59M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

/tmp/ipykernel_1870/2666396875.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embed_dim   = embed_model.get_sentence_embedding_dimension()


In [7]:
questions_emb_path = Path("questions_embeddings.npy")

if questions_emb_path.exists():
    questions_embeddings = np.load(str(questions_emb_path))
else:
    questions_embeddings = embed_model.encode(
        [row["query"] for row in questions],
        prompt_name="search_query",
        batch_size=128,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    np.save(str(questions_emb_path), questions_embeddings)


In [8]:
q_id_to_idx = {row["q_id"]: i for i, row in enumerate(questions)}

def retrieve(
    query: str,
    q_id: str,
    top_k: int = 7,
    min_score: float = 0.05,
) -> List[Tuple[Dict, float]]:
    q_lemma  = lemmatize(query)
    q_tokens = q_lemma.split()

    bm25_scores = np.array(bm25_index.get_scores(q_tokens), dtype=np.float32)
    bm25_norm   = bm25_scores / (bm25_scores.max() + 1e-9)

    word_scores = cosine_similarity(tfidf_word.transform([q_lemma]), M_word)[0].astype(np.float32)
    char_scores = cosine_similarity(tfidf_char.transform([q_lemma]), M_char)[0].astype(np.float32)

    q_vec = questions_embeddings[q_id_to_idx[q_id]].reshape(1, -1).astype(np.float32)
    dense_scores_raw, dense_indices = faiss_index.search(q_vec, len(all_chunks))
    dense_scores = np.zeros(len(all_chunks), dtype=np.float32)
    dense_scores[dense_indices[0]] = dense_scores_raw[0]
    dense_norm = dense_scores / (dense_scores.max() + 1e-9)

    hybrid = 0.30 * bm25_norm + 0.15 * word_scores + 0.10 * char_scores + 0.45 * dense_norm

    results      = []
    seen_doc_ids = set()
    for idx in np.argsort(hybrid)[::-1][: top_k * 3]:
        score  = float(hybrid[idx])
        if score < min_score:
            break
        chunk  = all_chunks[idx]
        doc_id = chunk["doc_id"]
        if doc_id not in seen_doc_ids:
            seen_doc_ids.add(doc_id)
            results.append((chunk, score))
        if len(results) >= top_k:
            break

    return results


In [9]:
def build_context(
    retrieved: List[Tuple[Dict, float]],
    max_chars: int = 6000,
) -> str:
    if not retrieved:
        return ""

    parts = []
    total_chars = 0
    for i, (chunk, score) in enumerate(retrieved, 1):
        snippet = chunk["text"].strip()
        part    = f"[{i}] {snippet}"
        if total_chars + len(part) > max_chars:
            remaining = max_chars - total_chars - len(f"[{i}] ")
            if remaining > 50:
                parts.append(f"[{i}] {snippet[:remaining]}...")
            break
        parts.append(part)
        total_chars += len(part)

    return "\n\n".join(parts)


In [10]:
RAG_ANSWER_PROMPT = """Ты - помощник Альфа-Банка. Отвечай на вопрос клиента строго на основе предоставленных фрагментов документов.

Правила:
- Используй только информацию из контекста ниже.
- Если в контексте нет ответа на вопрос - напиши: "По данному вопросу информация не найдена."
- Отвечай кратко и по существу: не более 2–3 предложений, если вопрос простой.
- Не придумывай факты. Используй факты из контекста и опирайся на свои знания только в том случае, когда абсолютно точно уверен в их корректности и релевантности
- Отвечай на русском языке.
- Не упоминай "Фрагмент", "контекст", "документ" в ответе - отвечай как живой сотрудник банка.

Контекст:
{{ context }}

Вопрос клиента: {{ query }}

Ответ:"""

NO_CONTEXT_PROMPT = """Ты - помощник Альфа-Банка. Клиент задал вопрос, но релевантной информации в базе знаний не найдено.

Вопрос клиента: {{ query }}

Вежливо сообщи клиенту, что по его вопросу информация не найдена, и предложи обратиться в поддержку Альфа-Банка.

Отвечай кратко, 1–2 предложения.

Ответ:"""

prompt_rag    = Template(RAG_ANSWER_PROMPT)
prompt_no_ctx = Template(NO_CONTEXT_PROMPT)


In [11]:
client = openai.OpenAI(
    api_key="EMPTY",
    base_url="http://localhost:8000/v1", # локальный сервинг геммы на vllm
)


def call_model(prompt_text: str) -> str:
    response = client.chat.completions.create(
        model="google/gemma-4-31b",
        max_tokens=300,
        temperature=0.1,
        messages=[{"role": "user", "content": prompt_text}],
    )
    return response.choices[0].message.content.strip()


def generate_answer(query: str, context: str) -> str:
    if not context:
        prompt_text = prompt_no_ctx.render(query=query)
    else:
        prompt_text = prompt_rag.render(query=query, context=context)
    return call_model(prompt_text)


In [12]:
def rag_pipeline(query: str, q_id: str) -> str:
    retrieved = retrieve(query, q_id)
    context   = build_context(retrieved)
    return generate_answer(query, context)


In [13]:
submission_path = Path("submission.csv")

results: List[Dict] = []
errors:  List[Dict] = []
done_ids: set = set()

if submission_path.exists():
    with open(submission_path, "r", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row.get("answer_new", "").strip():
                results.append({"q_id": row["q_id"], "answer_new": row["answer_new"]})
                done_ids.add(row["q_id"])

remaining = [row for row in questions if row["q_id"] not in done_ids]

for i, row in enumerate(tqdm(remaining)):
    q_id  = row["q_id"]
    query = row["query"]

    try:
        answer = rag_pipeline(query, q_id)
    except Exception as e:
        answer = ""
        errors.append({"q_id": q_id, "error": str(e)})

    results.append({"q_id": q_id, "answer_new": answer})
    time.sleep(0.05)

    if (i + 1) % 50 == 0 or (i + 1) == len(remaining):
        with open(submission_path, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["q_id", "answer_new"])
            writer.writeheader()
            writer.writerows(results)

if errors:
    print(f"ошибок: {len(errors)}, первые: {errors[:3]}")


100%|██████████| 6977/6977 [1:41:23<00:00,  1.14s/it]


ошибок: 1, первые: [{'q_id': '1629', 'error': 'list index out of range'}]
